# Descargando imagenes desde Alaska Satellite Facility

<p align="center">
  <img src="./assets/asf-logo-blue-nav.png" alt="ASF Logo" width="200"/>
</p>

El objetivo de este archivo es presentar un tutorial de como descargar datos desde el Data Vertex desde su API utilizando Python. Esta forma facilita las descargas masivas de datos. Este jupyter no corresponde a un tutorial de Python, no representa un tutorial oficial del ASF y tampoco un tutorial sobre sensores satelitales.

Los codigos presentados aqui son para casos de uso basico y se recomienda customizarlos para cumplir con las necesidades de cada proyecto, para ello se puede visitar [ASF SAR Data Search Manual](https://docs.asf.alaska.edu/) para mayor información.

Autor: Javier Norambuena Leiva

## Preparativos

Para descargar datos (tanto desde la API como de la interfaz web) es *obligatorio* el tener una cuenta en el [EarthData](https://urs.earthdata.nasa.gov/) de la NASA.

El siguiente paso es tener Python3 instalado en el equipo y un sistema de manejo de paquetes (CONDA o pip o poetry) que nos permita instalar diferentes librerias. La que necesitamos para conectarnos con el servicio de Data Vertex, necesitamos instalar:
- [asf_search](https://docs.asf.alaska.edu/asf_search/basics/)


## Construccion de la consulta

Para construir una consulta se necesitan definir ciertos detalles:
- El poligono: Representa el area de interes que se quiere estudiar, por simplicidad se puede utilizar el poligono generado por la misma pagina del DataVertex, ya que, viene en el formato que la API solicita.
- El dataset: Corresponde a la mision que uno quiere descargar (Sentinel 1, Radarsat1, etc), las opciones disponibles son: NISAR, SENTINEL-1, SLC-BURST, OPERA-S1, ALOS PALSAR, ALOS AVNIR-2, SIR-C, ARIA S1 GUNW, SMAP, UAVSAR, RADARSAT-1, ERS, JERS-1, AIRSAR, SEASAT.
  Para llamar alguna de ellas, se tiene que utilizar un string con el texto, por ejemplo, para ALOS PALSAR debe ser "ALOS PALSAR".
- Intervalo de fechas: Son las fechas en las que se necesitan las imagenes, se tiene que definir una fecha de inicio una de termino usando el formato "YYYY-MM-DDT00:00UTC".
- Nivel de procesamiento: Las imagenes vienen en diferentes niveles de procesamiento como L0 o L1, esto depende del conjunto de datos, por lo que se recomienda revisar la seccion ProcessingLevel dentro de [keywords](https://docs.asf.alaska.edu/api/keywords/).
- Beam mode: Los intrumentos dentro de cada mision, poseen diferentes modos de obtencion de datos, por lo tanto, se recomienda ver las opciones disponibles en la seccion beamMode dentro de [keywords](https://docs.asf.alaska.edu/api/keywords/).

Para todos los casos, dentro de la fucion **asf.geo_search**, todos estos campos deben ser entragados como strings.

In [ ]:
import asf_search as asf

pol = ##Poligono del area de interes
dataset = ## Seleccionar alguna de las opciones
level = ##Nivel
mode = ##beamMode
fecha_inicio = #'1995-01-01T00:00UTC'
fecha_termino = #'2014-01-01T00:00UTC'

results = asf.geo_search(
    dataset= dataset,
    intersectsWith= pol,
    start= fecha_inicio,
    end= fecha_termino,
    maxResults= 10000,
    processingLevel=level,
    beamMode= beam,
)


Dentro de la variable *results*, se encuentran todos las imagenes encontradas, en misiones antiguas es esperable que no se encuentren datos ya que estas operaban *on-demand*. Tambiem, es importante aclarar que las imagenes que se indican en esa variable son aquellas que intersectan el poligono, entonces, es posible que **varias imagenes no contengan al poligono completo** y se descargan en su totalidad, por lo tanto estas pueden consumir espacio importante del disco.

## Descargar y guardar
Una vez que tenemos el conjunto de datos que se quieren descargar, para ver el numero de imagenes se puede usar *print(len(results))*, el siguiente paso es logearse al servicio de **EarthData**, para ello, se tiene que escribir en las variables *user* y *password*, el usuario y contraseña respectivamente. Es importante aclarar que esta forma es funcional mas no segura, no es recomendable escribir las credenciales en codigo, pero para uso interno es suficiente.

Una vez ingresado, se va a ir iterando en cada una de las imagenes, en la variables **output_dir** se define donde se van a guardar los resultados.

In [ ]:
import os
import time

user = ##Usuario con el que se logea al EarthData
password = ##Contraseña de la cuenta

session = asf.ASFSession().auth_with_creds(user, password)

output_dir = './radarsat_data'
os.makedirs(output_dir, exist_ok=True)

for r in results:
    print(r.properties['sceneName'])
    r.download(path=output_dir, session=session)
    time.sleep(1)

## Ejemplos

### Sentinel 1 para volcan Melimoyu, Región de Aysén, Chile
Para este ejemplo se descargaran datos Sentinel 1 High Resolution del volcan Melimoyu, considerando una polarizacion *VV* en el periodo 2016-2020.

<p align="center">
  <img src="./assets/melimoyu.jpg" alt="Melimoyu photo" width="400"/>
</p>

Foto extraida desde [wikipedia](https://es.wikipedia.org/wiki/Volc%C3%A1n_Melimoyu).

En varias de las opciones, se pueden usar listas mas que string unicos en las solicitudes, lo cual permite complejizar, aca mezclamos dos modos, *Dual Polarization* y *Single Polarization*.

In [ ]:
import asf_search as asf
import os
import time

#Campos de busqueda
dataset = "SENTINEL-1"
level = ["GRD_HS", "GRD_HD"]
mode = "IW"
polarization = ["VV", "VV+VH", "Dual VV"]
fecha_inicio = '2016-01-01T00:00UTC'
fecha_termino = '2020-01-01T00:00UTC'
pol = 'POLYGON((-72.953 -44.1162,-72.7553 -44.1162,-72.7553 -44.0164,-72.953 -44.0164,-72.953 -44.1162))'

#Busqueda dentro de los datos de ASF
results = asf.geo_search(
    dataset= dataset,
    intersectsWith= pol,
    start= fecha_inicio,
    end= fecha_termino,
    maxResults= 10000,
    processingLevel= level,
    beamMode= mode,
    polarization= polarization
)

#Credeciales del usuario
user = ##Usuario con el que se logea al EarthData
password = ##Contraseña de la cuenta
session = asf.ASFSession().auth_with_creds(user, password)

#Guardar datos
output_dir = './sentinel1_data'
os.makedirs(output_dir, exist_ok=True)

for r in results:
    print(r.properties['sceneName'])
    r.download(path=output_dir, session=session)
    time.sleep(1)

### Radarsat 1 para glaciar Calluqueo, Región de Aysén, Chile
Para este ejemplo se descargaran datos RADARSAR procesados de nivel *L1* del glaciar Calluqueo considerando modo *Standard* entre los años 1990 y 2000.

<p align="center">
  <img src="./assets/calluqueo.jpg" alt="Calluqueo photo" width="400"/>
</p>

Foto extraida desde [Ladera Sur](https://laderasur.com/destino/calluqueo-un-glaciar-de-facil-acceso-en-cochrane/).

In [ ]:
import asf_search as asf
import os
import time

#Campos de busqueda
dataset = "RADARSAT-1"
level = ["L1"]
mode = "Standard"
fecha_inicio = '1990-01-01T00:00UTC'
fecha_termino = '2000-01-01T00:00UTC'
pol = 'POLYGON((-72.5148 -47.6095,-72.4144 -47.6095,-72.4144 -47.5696,-72.5148 -47.5696,-72.5148 -47.6095))'

#Busqueda dentro de los datos de ASF
results = asf.geo_search(
    dataset= dataset,
    intersectsWith= pol,
    start= fecha_inicio,
    end= fecha_termino,
    maxResults= 10000,
    processingLevel= level,
    beamMode= mode,
)

#Credeciales del usuario
user = ##Usuario con el que se logea al EarthData
password = ##Contraseña de la cuenta
session = asf.ASFSession().auth_with_creds(user, password)

#Guardar datos
output_dir = './radarsat1_data'
os.makedirs(output_dir, exist_ok=True)

for r in results:
    print(r.properties['sceneName'])
    r.download(path=output_dir, session=session)
    time.sleep(1)

### ERS para la cordillera Sentinel, Montes Ellsworth, Antártica
Para este ejemplo se descargaran datos ERS de la cordillera Sentinel de nivel *L1* considerando modo *Standard* entre los años 1990 y 2000.

<p align="center">
  <img src="./assets/montesellsworth.jpg" alt="ME photo" width="300"/>
</p>

Foto extraida desde [wikipedia](https://es.wikipedia.org/wiki/Montes_Ellsworth).

In [ ]:
import asf_search as asf
import os
import time

#Campos de busqueda
dataset = "ERS"
level = ["L1"]
mode = "Standard"
fecha_inicio = '1990-01-01T00:00UTC'
fecha_termino = '2000-01-01T00:00UTC'
pol = 'POLYGON((-88.4547 -79.0455,-82.5933 -79.0455,-82.5933 -77.2379,-88.4547 -77.2379,-88.4547 -79.0455))'

#Busqueda dentro de los datos de ASF
results = asf.geo_search(
    dataset= dataset,
    intersectsWith= pol,
    start= fecha_inicio,
    end= fecha_termino,
    maxResults= 10000,
    processingLevel= level,
    beamMode= mode,
)

#Credeciales del usuario
user = ##Usuario con el que se logea al EarthData
password = ##Contraseña de la cuenta
session = asf.ASFSession().auth_with_creds(user, password)

#Guardar datos
output_dir = './ers_data'
os.makedirs(output_dir, exist_ok=True)

for r in results:
    print(r.properties['sceneName'])
    r.download(path=output_dir, session=session)
    time.sleep(1)

### ALOS PALSAR para la Gran Campo Nevado, Región de Magallanes, Chile
Para este ejemplo se descargaran datos de DEM de la misión ALOS PALSAR del Gran Campo Nevado entre los años 2010 y 2011.

<p align="center">
  <img src="./assets/grancamponevado.jpg" alt="GCN photo" width="400"/>
</p>

Foto extraida desde el siguiente [blog](https://www.geologie.uni-freiburg.de/root/people/studis/hohnerm/Bilder_Patagonien/Patagonien_Bilder_sp.html).

In [ ]:
import asf_search as asf
import os
import time

#Campos de busqueda
dataset = "ALOS PALSAR"
level = ["RTC_HI_RES"]
fecha_inicio = '2010-01-01T00:00UTC'
fecha_termino = '2011-01-01T00:00UTC'
pol = 'POLYGON((-73.2399 -52.9234,-72.8904 -52.9234,-72.8904 -52.7279,-73.2399 -52.7279,-73.2399 -52.9234))'

#Busqueda dentro de los datos de ASF
results = asf.geo_search(
    dataset= dataset,
    intersectsWith= pol,
    start= fecha_inicio,
    end= fecha_termino,
    maxResults= 10000,
    processingLevel= level,
)

#Credeciales del usuario
user = ##Usuario con el que se logea al EarthData
password = ##Contraseña de la cuenta
session = asf.ASFSession().auth_with_creds(user, password)

#Guardar datos
output_dir = './alospalsar_data'
os.makedirs(output_dir, exist_ok=True)

for r in results:
    print(r.properties['sceneName'])
    r.download(path=output_dir, session=session)
    time.sleep(1)